# Interactive EDA: BERTopic Model with Radway Mappings

This notebook provides comprehensive interactive exploration of the final BERTopic model with:
- **Taxonomy mappings** (Stage 2): Main and secondary categories, groups, confidence scores
- **Radway narrative function mappings** (Stage 3): Main functions, phases, confidence, rationales
- **Topic representations**: Keywords, labels, and metadata
- **Statistical summaries**: Distributions, cross-tabulations, and relationships

## Navigation

1. **Setup & Data Loading**: Import libraries and load the model
2. **Data Overview**: Summary statistics and data structure
3. **Taxonomy Analysis**: Distribution and patterns in taxonomy categories
4. **Radway Analysis**: Distribution and patterns in Radway narrative functions
5. **Cross-Analysis**: Relationships between taxonomy and Radway mappings
6. **Interactive Exploration**: Filtering, searching, and custom queries
7. **Topic Deep Dives**: Detailed examination of specific topics

In [1]:
# Helper function for robust Plotly figure display
def show_plotly_fig(fig, save_html=False, output_dir=None):
    """Display Plotly figure with fallback options.
    
    Args:
        fig: Plotly figure object
        save_html: If True, also save as HTML file
        output_dir: Directory to save HTML (if save_html=True)
    """
    try:
        # Try to show in notebook
        fig.show()
    except (ValueError, ImportError) as e:
        # Fallback: save to HTML and display message
        if output_dir is None:
            output_dir = project_root / "results" / "stage09_category_mapping" / "stage3_radway_functions" / "eda"
        output_dir.mkdir(parents=True, exist_ok=True)
        
        html_path = output_dir / f"plot_{hash(str(fig.layout.title.text if fig.layout.title else 'figure'))}.html"
        fig.write_html(str(html_path))
        print(f"⚠️  Could not display figure in notebook. Saved to: {html_path}")
        print(f"   Open this file in a browser to view the interactive plot.")
        
        if save_html:
            return html_path
    else:
        # If show() succeeded, optionally save HTML too
        if save_html:
            if output_dir is None:
                output_dir = project_root / "results" / "stage09_category_mapping" / "stage3_radway_functions" / "eda"
            output_dir.mkdir(parents=True, exist_ok=True)
            html_path = output_dir / f"plot_{hash(str(fig.layout.title.text if fig.layout.title else 'figure'))}.html"
            fig.write_html(str(html_path))
            return html_path


## 1. Setup & Imports

In [2]:
import json
import os
import sys
from pathlib import Path
from typing import Any, Dict, List

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from bertopic import BERTopic

# Find project root by looking for marker files
def find_project_root():
    """Find the project root directory by looking for marker files."""
    # Start from current working directory
    cwd = Path(os.getcwd()).resolve()
    
    # Look for project root markers
    markers = ['README.md', 'requirements.txt', 'SCIENTIFIC_README.md', 'Makefile']
    
    # Try going up from current directory
    current = cwd
    for _ in range(10):  # Max 10 levels up
        # Check if this looks like the project root
        has_markers = any((current / marker).exists() for marker in markers)
        has_src = (current / 'src').exists()
        has_notebooks = (current / 'notebooks').exists()
        
        if has_markers and has_src and has_notebooks:
            return current
        
        # If we're in a 'notebooks' subdirectory, go up to project root
        if 'notebooks' in current.parts:
            parts = list(current.parts)
            idx = parts.index('notebooks')
            candidate = Path(*parts[:idx])
            if (candidate / 'src').exists() and (candidate / 'notebooks').exists():
                return candidate
        
        if current == current.parent:
            break
        current = current.parent
    
    # Fallback: use known absolute path
    fallback = Path("/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor")
    if (fallback / 'src').exists() and (fallback / 'notebooks').exists():
        return fallback
    
    # Last resort: assume we're in notebooks/07_analysis, go up 2 levels
    return cwd.parent.parent

# Get project root
project_root = find_project_root()
sys.path.insert(0, str(project_root))

# Import with fallback to absolute path if needed
try:
    from src.stage06_topic_exploration.explore_retrained_model import (
        DEFAULT_BASE_DIR,
        DEFAULT_EMBEDDING_MODEL,
    )
except ImportError:
    # If import fails, try absolute path
    project_root = Path("/home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor")
    if str(project_root) not in sys.path:
        sys.path.insert(0, str(project_root))
    from src.stage06_topic_exploration.explore_retrained_model import (
        DEFAULT_BASE_DIR,
        DEFAULT_EMBEDDING_MODEL,
    )

# Set style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# Enable inline plotting
%matplotlib inline
%config InlineBackend.figure_format = 'retina'

print(f"Project root: {project_root}")
print(f"Default base dir: {DEFAULT_BASE_DIR}")
print(f"Default embedding model: {DEFAULT_EMBEDDING_MODEL}")


[2025-12-13 21:42:39.847] [CUML] [info] build_algo set to brute_force_knn because random_state is given
✅ RAPIDS (cuML) is available and functional
Project root: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor
Default base dir: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/models/retrained
Default embedding model: paraphrase-MiniLM-L6-v2


In [3]:
# Suppress tqdm/ipywidgets warning (harmless - just means progress bars won't show in notebook)
# This warning appears because ipywidgets is not installed, but it doesn't affect functionality
import warnings
warnings.filterwarnings('ignore', message='.*IProgress not found.*')
warnings.filterwarnings('ignore', category=UserWarning, module='tqdm.auto')

## 2. Helper Functions

In [4]:
def load_labels_metadata(
    project_root: Path,
    labels_filename: str = "labels_pos_openrouter_mistralai_Mistral-Nemo-Instruct-2407_romance_aware_paraphrase-MiniLM-L6-v2.json"
) -> dict[int, dict[str, Any]]:
    """Load full metadata from labels JSON file (Stage 8).
    
    Args:
        project_root: Project root directory
        labels_filename: Name of the labels JSON file
        
    Returns:
        Dictionary mapping topic_id to full metadata dict
    """
    labels_path = project_root / "results" / "stage08_llm_labeling" / labels_filename
    
    if not labels_path.exists():
        # Try to find any labels file
        labels_dir = project_root / "results" / "stage08_llm_labeling"
        if labels_dir.exists():
            json_files = list(labels_dir.glob("labels_*.json"))
            if json_files:
                labels_path = json_files[0]
                print(f"Using labels file: {labels_path.name}")
            else:
                print(f"Warning: No labels JSON file found in {labels_dir}")
                return {}
        else:
            print(f"Warning: Labels directory not found: {labels_dir}")
            return {}
    
    print(f"Loading labels metadata from: {labels_path}")
    with open(labels_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    
    # Convert string keys to int and preserve full structure
    metadata: dict[int, dict[str, Any]] = {}
    for topic_id_str, topic_data in data.items():
        topic_id = int(topic_id_str)
        if isinstance(topic_data, dict):
            metadata[topic_id] = topic_data.copy()
        elif isinstance(topic_data, str):
            # Simple label-only format, convert to dict
            metadata[topic_id] = {"label": topic_data}
    
    print(f"✓ Loaded metadata for {len(metadata)} topics")
    return metadata


In [5]:
def load_model_with_radway(
    base_dir: Path = DEFAULT_BASE_DIR,
    embedding_model: str = DEFAULT_EMBEDDING_MODEL,
    model_suffix: str = "_with_radway_mappings",
    stage_subfolder: str = "stage09_category_mapping",
) -> BERTopic:
    """Load the final model with Radway mappings."""
    model_path = base_dir / embedding_model / stage_subfolder / f"model_1{model_suffix}"
    print(f"Loading model from: {model_path}")
    if not model_path.exists():
        raise FileNotFoundError(f"Model not found at {model_path}")
    return BERTopic.load(str(model_path))


def extract_all_fields(model: BERTopic, labels_metadata: dict[int, dict[str, Any]] | None = None) -> pd.DataFrame:
    """Extract all available fields from the model into a DataFrame.
    
    Args:
        model: BERTopic model instance
        labels_metadata: Optional dictionary mapping topic_id to labels metadata from JSON file
    """
    rows = []

    # Get topic IDs (exclude outlier -1)
    topic_ids = [
        tid
        for tid in model.topic_representations_.keys()
        if tid != -1
    ]

    for topic_id in sorted(topic_ids):
        row = {"topic_id": topic_id}

        # Topic representations (keywords)
        if hasattr(model, "topic_representations_") and topic_id in model.topic_representations_:
            keywords = model.topic_representations_[topic_id]
            row["keywords"] = ", ".join([kw[0] for kw in keywords[:10]])  # Top 10 keywords
            row["num_keywords"] = len(keywords)
            row["all_keywords"] = [kw[0] for kw in keywords]  # Store all for filtering

        # Load labels and metadata from JSON file (Stage 8) - preferred source
        if labels_metadata and topic_id in labels_metadata:
            meta = labels_metadata[topic_id]
            row["label"] = meta.get("label", None)
            row["scene_summary"] = meta.get("scene_summary", None)
            row["primary_categories"] = ", ".join(meta.get("primary_categories", [])) if meta.get("primary_categories") else None
            row["secondary_categories"] = ", ".join(meta.get("secondary_categories", [])) if meta.get("secondary_categories") else None
            row["label_is_noise"] = meta.get("is_noise", False)
            row["label_rationale"] = meta.get("rationale", None)
        else:
            # Fallback to model's custom labels if JSON not available
            if hasattr(model, "topic_labels_") and model.topic_labels_:
                row["label"] = model.topic_labels_.get(topic_id, None)
            elif hasattr(model, "custom_labels_") and model.custom_labels_:
                if isinstance(model.custom_labels_, list):
                    # List format: index corresponds to topic_id (with -1 at index 0)
                    label_idx = topic_id + 1 if topic_id >= 0 else 0
                    row["label"] = model.custom_labels_[label_idx] if label_idx < len(model.custom_labels_) else None
                elif isinstance(model.custom_labels_, dict):
                    row["label"] = model.custom_labels_.get(topic_id, None)
                else:
                    row["label"] = None
            else:
                row["label"] = None
            row["scene_summary"] = None
            row["primary_categories"] = None
            row["secondary_categories"] = None
            row["label_is_noise"] = None
            row["label_rationale"] = None
        # Taxonomy mappings (Stage 2)
        if hasattr(model, "topic_taxonomy_") and model.topic_taxonomy_:
            taxonomy = model.topic_taxonomy_.get(topic_id, {})
            row.update({
                "taxonomy_main_id": taxonomy.get("main_category_id"),
                "taxonomy_main_name": taxonomy.get("main_category_name"),
                "taxonomy_main_group": taxonomy.get("main_category_group"),
                "taxonomy_secondary_id": taxonomy.get("secondary_category_id"),
                "taxonomy_secondary_name": taxonomy.get("secondary_category_name"),
                "taxonomy_secondary_group": taxonomy.get("secondary_category_group"),
                "taxonomy_confidence": taxonomy.get("confidence"),
                "taxonomy_is_noise": taxonomy.get("is_noise", False),
            })
        else:
            row.update({
                "taxonomy_main_id": None,
                "taxonomy_main_name": None,
                "taxonomy_main_group": None,
                "taxonomy_secondary_id": None,
                "taxonomy_secondary_name": None,
                "taxonomy_secondary_group": None,
                "taxonomy_confidence": None,
                "taxonomy_is_noise": None,
            })

        # Radway mappings (Stage 3)
        if hasattr(model, "topic_radway_") and model.topic_radway_:
            radway = model.topic_radway_.get(topic_id, {})
            row.update({
                "radway_main_id": radway.get("radway_main_id"),
                "radway_main_name": radway.get("radway_main_name"),
                "radway_secondary_id": radway.get("radway_secondary_id"),
                "radway_phase": radway.get("radway_phase"),
                "radway_phase_name": radway.get("radway_phase_name"),
                "radway_is_none": radway.get("radway_is_none", False),
                "radway_confidence": radway.get("radway_confidence"),
                "radway_rationale": radway.get("radway_rationale"),
            })
        else:
            row.update({
                "radway_main_id": None,
                "radway_main_name": None,
                "radway_secondary_id": None,
                "radway_phase": None,
                "radway_phase_name": None,
                "radway_is_none": None,
                "radway_confidence": None,
                "radway_rationale": None,
            })

        rows.append(row)

    return pd.DataFrame(rows)

## 3. Load Model and Extract Data

In [6]:
# Load the model
model = load_model_with_radway()

# Load labels metadata from JSON file (Stage 8)
labels_metadata = load_labels_metadata(project_root)

# Extract all fields into DataFrame (including labels metadata)
df = extract_all_fields(model, labels_metadata=labels_metadata)

print(f"✓ Loaded model and extracted data for {len(df)} topics")
print(f"\nDataFrame shape: {df.shape}")
print(f"\nColumns ({len(df.columns)}):")
for col in df.columns:
    print(f"  - {col}")

Loading model from: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/models/retrained/paraphrase-MiniLM-L6-v2/stage09_category_mapping/model_1_with_radway_mappings
Loading labels metadata from: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage08_llm_labeling/labels_pos_openrouter_mistralai_Mistral-Nemo-Instruct-2407_romance_aware_paraphrase-MiniLM-L6-v2.json
✓ Loaded metadata for 361 topics
✓ Loaded model and extracted data for 368 topics

DataFrame shape: (368, 26)

Columns (26):
  - topic_id
  - keywords
  - num_keywords
  - all_keywords
  - label
  - scene_summary
  - primary_categories
  - secondary_categories
  - label_is_noise
  - label_rationale
  - taxonomy_main_id
  - taxonomy_main_name
  - taxonomy_main_group
  - taxonomy_secondary_id
  - taxonomy_secondary_name
  - taxonomy_secondary_group
  - taxonomy_confidence
  - taxonomy_is_noise
  - radway_main_id
  - radway_main_name
 

In [7]:
# Display first few rows
df.head(10)

,topic_id,keywords,num_keywords,all_keywords,label,scene_summary,primary_categories,secondary_categories,label_is_noise,label_rationale,...,taxonomy_confidence,taxonomy_is_noise,radway_main_id,radway_main_name,radway_secondary_id,radway_phase,radway_phase_name,radway_is_none,radway_confidence,radway_rationale
0,0,"ita, ia, need, say, want, going, dona, means, ...",27,"[ita, ia, need, say, want, going, dona, means,...",Negotiating Deal,The couple discusses terms and expectations fo...,"relationship_conflict, domestic_life","setting:living_room, activity:discussion",False,"The top keywords 'means', 'deal', 'today' and ...",...,medium,False,none,None of the above,None,NA,Not a narrative function,True,high,This topic is about the hero's work and busine...
1,1,"kissed, hips, tongue, breasts, mouth, body, ar...",27,"[kissed, hips, tongue, breasts, mouth, body, a...",Intimate Breast And Nipple Kissing,He gently kisses her breasts and nipples while...,"physical_affection, sexual_content","setting:bedroom, activity:kissing",False,"The top keywords 'breasts', 'mouth', 'nipples'...",...,high,False,R12,Heroine responds sexually and emotionally,None,III,Commitment & Restoration,False,high,The topic's representative snippets all depict...
2,2,"clit, pussy, tongue, hips, legs, mouth, neck, ...",27,"[clit, pussy, tongue, hips, legs, mouth, neck,...",Clitoral Stimulation During Foreplay,"She spreads her legs, allowing him to run his ...","sexual_content, romance_core","setting:bedroom, activity:oral_sex, sexual:cli...",False,"The top keywords 'clit', 'tongue', 'hips', 'le...",...,high,False,R12,Heroine responds sexually and emotionally,None,III,Commitment & Restoration,False,high,This topic focuses on explicit sexual acts bet...
3,3,"dinner, eat, food, lunch, breakfast, bakery, c...",27,"[dinner, eat, food, lunch, breakfast, bakery, ...",Dinner Invitation,"He invites her to dinner, suggesting a restaur...","romance_core, social_setting","setting:restaurant, activity:invitation",False,The top keyword 'dinner' and the snippets 'hav...,...,high,False,none,None of the above,None,NA,Not a narrative function,True,high,The topic keywords and representative snippets...
4,4,"seen, felt, ia, wanted, hea, experienced, love...",27,"[seen, felt, ia, wanted, hea, experienced, lov...",Unclear Relationship Feelings,The couple expresses confusion about their rel...,"romance_core, relationship_conflict","setting:home, activity:conversation",False,"The top keywords 'relationship', 'feelings', a...",...,medium,False,R3,Hero responds ambiguously to heroine,None,I,Initial Conflict & Isolation,False,medium,The topic revolves around characters expressin...
5,5,"gone, gotten, given, week, trying, years, lear...",27,"[gone, gotten, given, week, trying, years, lea...",Long-term Relationship Struggles,"She reflects on their years together, wonderin...","romance_core, relationship_conflict","setting:home, relationship:long_term",False,"The keywords 'years', 'times', and 'things' in...",...,high,False,R10,Heroine reinterprets hero's behaviour as resul...,None,II,Turning Point & Recognition,False,medium,The topic revolves around the heroine's intern...
6,6,"wine, sip, bottle, drunk, whiskey, waiter, sco...",27,"[wine, sip, bottle, drunk, whiskey, waiter, sc...",Wine-drinking At Table,"The couple sits at a table, sipping wine from ...","social_setting, physical_affection","setting:table, activity:drinking_wine",False,"The top keywords 'wine', 'sip', 'bottle', and ...",...,high,False,none,None of the above,None,NA,Not a narrative function,True,high,The topic keywords and representative snippets...
7,7,"didn, won, therea, doesn, aren, thata, isn, ma...",27,"[didn, won, therea, doesn, aren, thata, isn, m...",Relationship Ambiguity Conversation,"The couple sits together, struggling to expres...","romance_core, relationship_conflict","setting:living_room, activity:conversation",False,"The keywords 'problem', 'deal', 'idea', 'point...",...,medium,False,R3,Hero responds ambiguously

In [8]:
# Basic info about the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 368 entries, 0 to 367
Data columns (total 26 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   topic_id                  368 non-null    int64 
 1   keywords                  368 non-null    object
 2   num_keywords              368 non-null    int64 
 3   all_keywords              368 non-null    object
 4   label                     368 non-null    object
 5   scene_summary             361 non-null    object
 6   primary_categories        351 non-null    object
 7   secondary_categories      351 non-null    object
 8   label_is_noise            361 non-null    object
 9   label_rationale           351 non-null    object
 10  taxonomy_main_id          361 non-null    object
 11  taxonomy_main_name        361 non-null    object
 12  taxonomy_main_group       361 non-null    object
 13  taxonomy_secondary_id     135 non-null    object
 14  taxonomy_secondary_name   

## 4. Summary Statistics

In [48]:
summary = {
    "total_topics": len(df),
    "topics_with_labels": df["label"].notna().sum(),
    "topics_with_taxonomy": df["taxonomy_main_id"].notna().sum(),
    "topics_with_radway": df["radway_main_id"].notna().sum(),
    "topics_with_radway_function": (df["radway_is_none"] == False).sum(),
    "topics_with_radway_none": (df["radway_is_none"] == True).sum(),
    "unique_taxonomy_categories": df["taxonomy_main_name"].nunique(),
    "unique_taxonomy_groups": df["taxonomy_main_group"].nunique(),
    "unique_radway_functions": df[df["radway_is_none"] == False]["radway_main_name"].nunique(),
    "unique_radway_phases": df["radway_phase_name"].nunique(),
}

print("=" * 80)
print("OVERALL STATISTICS")
print("=" * 80)
for key, value in summary.items():
    print(f"{key:.<50} {value}")

# Create a summary DataFrame for better visualization
summary_df = pd.DataFrame([summary]).T
summary_df.columns = ["Value"]
summary_df

OVERALL STATISTICS
total_topics...................................... 368
topics_with_labels................................ 368
topics_with_taxonomy.............................. 361
topics_with_radway................................ 361
topics_with_radway_function....................... 260
topics_with_radway_none........................... 108
unique_taxonomy_categories........................ 30
unique_taxonomy_groups............................ 9
unique_radway_functions........................... 13
unique_radway_phases.............................. 4


,Value
total_topics,368
topics_with_labels,368
topics_with_taxonomy,361
topics_with_radway,361
topics_with_radway_function,260
topics_with_radway_none,108
unique_taxonomy_categories,30
unique_taxonomy_groups,9
unique_radway_functions,13
unique_radway_phases,4


## 5. Taxonomy Distribution Analysis

In [10]:
# Main category distribution
print("Top 20 Main Categories:")
print(df["taxonomy_main_name"].value_counts().head(20))

Top 20 Main Categories:
taxonomy_main_name
Conflict, Distance & Breakup Threats               58
Bonding, Everyday Intimacy & Growth                43
Negative Emotions & Distress                       38
Violence, Threats & Coercion                       22
Domestic Spaces & Routines                         21
Explicit Sexual Acts                               20
Kissing & Non-Explicit Affection                   19
Attraction & Sexual Tension                        18
Secrets, Misunderstandings & Hidden Information    17
Hero's Elite Work & Business World                 16
Positive Emotions & Security                       11
Public & Leisure Spaces                            11
Friends & Social Circles                            9
Ambivalence & Internal Conflict                     8
Family & Kinship                                    6
Beliefs, Values & Moral Reflection                  6
Exercise & Physical Activity                        6
Reconciliation, Commitments & HEA      

In [11]:
# Category group distribution
print("Category Group Distribution:")
print(df["taxonomy_main_group"].value_counts())

Category Group Distribution:
taxonomy_main_group
Relationship Trajectory (Main Couple)    126
Emotions, Cognition & Inner Life          63
Sexuality, Attraction & Intimacy          57
Spaces, Time, Activities & Objects        34
Work, Wealth, Status & Institutions       27
Conflict, Risk & Harm                     26
Social World Outside Couple               18
Embodied & Sensory Experience              9
Special                                    1
Name: count, dtype: int64


In [12]:
# Interactive visualization: Top main categories
top_main = df["taxonomy_main_name"].value_counts().head(20)
fig = px.bar(
    x=top_main.values,
    y=top_main.index,
    orientation='h',
    title="Top 20 Main Taxonomy Categories",
    labels={"x": "Count", "y": "Category"},
    height=600
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [13]:
# Category groups visualization
main_group_counts = df["taxonomy_main_group"].value_counts()
fig = px.bar(
    x=main_group_counts.index,
    y=main_group_counts.values,
    title="Distribution by Taxonomy Category Group",
    labels={"x": "Category Group", "y": "Count"},
    height=400
)
fig.update_xaxes(tickangle=45)
fig.show()

In [14]:
# Confidence distribution
conf_counts = df["taxonomy_confidence"].value_counts()
fig = px.pie(
    values=conf_counts.values,
    names=conf_counts.index,
    title="Taxonomy Confidence Distribution"
)
fig.show()

In [15]:
# Noise vs non-noise
noise_counts = df["taxonomy_is_noise"].value_counts()
fig = px.bar(
    x=["Non-Noise", "Noise"],
    y=[noise_counts.get(False, 0), noise_counts.get(True, 0)],
    title="Noise vs Non-Noise Topics",
    labels={"x": "Category", "y": "Count"},
    color=["Non-Noise", "Noise"],
    color_discrete_map={"Non-Noise": "lightblue", "Noise": "coral"}
)
fig.show()

## 6. Radway Narrative Function Distribution Analysis

In [16]:
# Main Radway function distribution
print("Main Radway Function Distribution:")
radway_main_counts = df["radway_main_name"].value_counts()
print(radway_main_counts)

Main Radway Function Distribution:
radway_main_name
None of the above                                                   108
Hero and heroine are physically or emotionally separated             57
Hero treats heroine tenderly                                         43
Heroine reacts antagonistically to the hero                          42
Heroine interprets hero's behaviour as purely sexual interest        31
Heroine's social identity is destroyed                               18
Heroine responds warmly to hero's tenderness                         17
Heroine responds sexually and emotionally                            15
Hero retaliates or punishes heroine                                  11
Hero responds ambiguously to heroine                                  7
Heroine responds with anger or coldness                               4
Heroine reinterprets hero's behaviour as result of previous hurt      3
Heroine's identity is restored                                        3
Hero declare

In [17]:
# Phase distribution
print("\nPhase Distribution:")
phase_counts = df["radway_phase_name"].value_counts()
print(phase_counts)


Phase Distribution:
radway_phase_name
Initial Conflict & Isolation    170
Not a narrative function        108
Turning Point & Recognition      63
Commitment & Restoration         20
Name: count, dtype: int64


In [18]:
# Interactive visualization: Radway functions (excluding "none")
radway_without_none = df[df["radway_is_none"] == False]["radway_main_name"].value_counts()
fig = px.bar(
    x=radway_without_none.values,
    y=radway_without_none.index,
    orientation='h',
    title="Radway Functions Distribution (excluding 'none')",
    labels={"x": "Count", "y": "Function"},
    height=600
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [19]:
# Phase distribution visualization
phase_counts = df["radway_phase_name"].value_counts()
fig = px.bar(
    x=phase_counts.index,
    y=phase_counts.values,
    title="Distribution by Narrative Phase",
    labels={"x": "Phase", "y": "Count"},
    height=400
)
fig.update_xaxes(tickangle=45)
fig.show()

In [20]:
# Radway confidence distribution
radway_conf_counts = df["radway_confidence"].value_counts()
fig = px.pie(
    values=radway_conf_counts.values,
    names=radway_conf_counts.index,
    title="Radway Confidence Distribution"
)
fig.show()

In [21]:
# None vs function distribution
none_counts = df["radway_is_none"].value_counts()
fig = px.bar(
    x=["Function", "None"],
    y=[none_counts.get(False, 0), none_counts.get(True, 0)],
    title="None vs Narrative Function",
    labels={"x": "Category", "y": "Count"},
    color=["Function", "None"],
    color_discrete_map={"Function": "lightgreen", "None": "coral"}
)
fig.show()

## 7. Cross-Tabulation Analysis

In [22]:
# Taxonomy group vs Radway phase
crosstab_group_phase = pd.crosstab(
    df["taxonomy_main_group"],
    df["radway_phase_name"],
    margins=True,
)
print("Taxonomy Group vs Radway Phase:")
print(crosstab_group_phase)

Taxonomy Group vs Radway Phase:
radway_phase_name                      Commitment & Restoration  \
taxonomy_main_group                                               
Conflict, Risk & Harm                                         0   
Embodied & Sensory Experience                                 0   
Emotions, Cognition & Inner Life                              0   
Relationship Trajectory (Main Couple)                         5   
Sexuality, Attraction & Intimacy                             14   
Social World Outside Couple                                   1   
Spaces, Time, Activities & Objects                            0   
Special                                                       0   
Work, Wealth, Status & Institutions                           0   
All                                                          20   

radway_phase_name                      Initial Conflict & Isolation  \
taxonomy_main_group                                                   
Conflict, Risk & Harm

In [23]:
# Interactive heatmap: Taxonomy group vs Radway phase
crosstab_group_phase_plot = pd.crosstab(
    df["taxonomy_main_group"],
    df["radway_phase_name"],
)
fig = px.imshow(
    crosstab_group_phase_plot,
    labels=dict(x="Radway Phase", y="Taxonomy Group", color="Count"),
    title="Taxonomy Group vs Radway Phase",
    aspect="auto",
    color_continuous_scale="YlOrRd"
)
fig.update_xaxes(tickangle=45)
fig.show()

In [24]:
# Top taxonomy categories vs Radway functions
top_taxonomy = df["taxonomy_main_name"].value_counts().head(10).index
top_radway = df[df["radway_is_none"] == False]["radway_main_name"].value_counts().head(10).index
df_filtered = df[
    (df["taxonomy_main_name"].isin(top_taxonomy))
    & (df["radway_main_name"].isin(top_radway))
]

if not df_filtered.empty:
    crosstab_cat_func = pd.crosstab(
        df_filtered["taxonomy_main_name"],
        df_filtered["radway_main_name"],
        margins=True,
    )
    print("Top Taxonomy Categories vs Top Radway Functions:")
    print(crosstab_cat_func)

Top Taxonomy Categories vs Top Radway Functions:
radway_main_name                                 Hero and heroine are physically or emotionally separated  \
taxonomy_main_name                                                                                          
Attraction & Sexual Tension                                                                      0          
Bonding, Everyday Intimacy & Growth                                                              0          
Conflict, Distance & Breakup Threats                                                            49          
Explicit Sexual Acts                                                                             0          
Hero's Elite Work & Business World                                                               1          
Kissing & Non-Explicit Affection                                                                 0          
Negative Emotions & Distress                                                   

In [25]:
# Interactive heatmap: Top taxonomy categories vs top Radway functions
if not df_filtered.empty:
    crosstab_cat_func_plot = pd.crosstab(
        df_filtered["taxonomy_main_name"],
        df_filtered["radway_main_name"],
    )
    fig = px.imshow(
        crosstab_cat_func_plot,
        labels=dict(x="Radway Function", y="Taxonomy Category", color="Count"),
        title="Top Taxonomy Categories vs Top Radway Functions",
        aspect="auto",
        color_continuous_scale="YlOrRd",
        height=600
    )
    fig.update_xaxes(tickangle=45)
    fig.show()

## 8. Interactive Exploration & Filtering

In [26]:
# Filter by taxonomy category
def filter_by_taxonomy_category(df, category_name):
    """Filter topics by taxonomy category name."""
    return df[df["taxonomy_main_name"] == category_name]

# Example: Filter by a specific category
# Replace with any category name from your data
example_category = df["taxonomy_main_name"].value_counts().index[0]
filtered_df = filter_by_taxonomy_category(df, example_category)
print(f"Topics in category '{example_category}': {len(filtered_df)}")
filtered_df[["topic_id", "keywords", "taxonomy_main_name", "radway_main_name", "radway_phase_name"]].head()


Topics in category 'Conflict, Distance & Breakup Threats': 58


,topic_id,keywords,taxonomy_main_name,radway_main_name,radway_phase_name
28,28,"dona, want, say, need, think, talk, really, an...","Conflict, Distance & Breakup Threats",Heroine reacts antagonistically to the hero,Initial Conflict & Isolation
32,32,"firmly, offense, answered, youa, say, flatly, ...","Conflict, Distance & Breakup Threats",Heroine reacts antagonistically to the hero,Initial Conflict & Isolation
37,37,"ridiculous, insane, bullshit, fool, dumbass, o...","Conflict, Distance & Breakup Threats",Hero and heroine are physically or emotionally...,Initial Conflict & Isolation
52,52,"sama, www, didn, event, mocking, muttered, ans...","Conflict, Distance & Breakup Threats",Heroine reacts antagonistically to the hero,Initial Conflict & Isolation
59,59,"tristana, smiles, replies, broadly, laughs, pu...","Conflict, Distance & Breakup Threats",Hero and heroine are physically or emotionally...,Initial Conflict & Isolation


In [27]:
# Filter by Radway function
def filter_by_radway_function(df, function_name, exclude_none=True):
    """Filter topics by Radway function name."""
    if exclude_none:
        df = df[df["radway_is_none"] == False]
    return df[df["radway_main_name"] == function_name]

# Example: Filter by a specific Radway function
radway_functions = df[df["radway_is_none"] == False]["radway_main_name"].unique()
if len(radway_functions) > 0:
    example_function = radway_functions[0]
    filtered_df = filter_by_radway_function(df, example_function)
    print(f"Topics with Radway function '{example_function}': {len(filtered_df)}")
    filtered_df[["topic_id", "keywords", "taxonomy_main_name", "radway_main_name", "radway_phase_name"]].head()


Topics with Radway function 'Heroine responds sexually and emotionally': 15


In [28]:
# Filter by Radway phase
def filter_by_radway_phase(df, phase_name):
    """Filter topics by Radway phase name."""
    return df[df["radway_phase_name"] == phase_name]

# Example: Filter by a specific phase
phases = df["radway_phase_name"].unique()
if len(phases) > 0:
    example_phase = phases[0]
    filtered_df = filter_by_radway_phase(df, example_phase)
    print(f"Topics in phase '{example_phase}': {len(filtered_df)}")
    filtered_df[["topic_id", "keywords", "taxonomy_main_name", "radway_main_name", "radway_phase_name"]].head()


Topics in phase 'Not a narrative function': 108


In [29]:
# Search topics by keyword
def search_topics_by_keyword(df, keyword, case_sensitive=False):
    """Search topics by keyword in keywords or labels."""
    if case_sensitive:
        mask = df["keywords"].str.contains(keyword, na=False) | df["label"].str.contains(keyword, na=False)
    else:
        mask = df["keywords"].str.contains(keyword, case=False, na=False) | df["label"].str.contains(keyword, case=False, na=False)
    return df[mask]

# Example: Search for topics containing a keyword
example_keyword = "love"  # Replace with any keyword
filtered_df = search_topics_by_keyword(df, example_keyword)
print(f"Topics containing '{example_keyword}': {len(filtered_df)}")
if len(filtered_df) > 0:
    filtered_df[["topic_id", "keywords", "label", "taxonomy_main_name", "radway_main_name"]].head(10)


Topics containing 'love': 13


In [30]:
# Filter by confidence levels
def filter_by_confidence(df, taxonomy_min=None, radway_min=None):
    """Filter topics by confidence thresholds."""
    mask = pd.Series([True] * len(df))
    if taxonomy_min is not None:
        mask = mask & (df["taxonomy_confidence"] >= taxonomy_min)
    if radway_min is not None:
        mask = mask & (df["radway_confidence"] >= radway_min)
    return df[mask]

# Example: Filter by high confidence
high_conf_df = filter_by_confidence(df, taxonomy_min="high", radway_min="high")
print(f"Topics with high confidence in both taxonomy and Radway: {len(high_conf_df)}")
high_conf_df[["topic_id", "keywords", "taxonomy_confidence", "radway_confidence"]].head()


Topics with high confidence in both taxonomy and Radway: 361


,topic_id,keywords,taxonomy_confidence,radway_confidence
0,0,"ita, ia, need, say, want, going, dona, means, ...",medium,high
1,1,"kissed, hips, tongue, breasts, mouth, body, ar...",high,high
2,2,"clit, pussy, tongue, hips, legs, mouth, neck, ...",high,high
3,3,"dinner, eat, food, lunch, breakfast, bakery, c...",high,high
4,4,"seen, felt, ia, wanted, hea, experienced, love...",medium,medium


## 9. Topic Deep Dives

In [31]:
def explore_topic(df, topic_id):
    """Display detailed information about a specific topic."""
    topic_data = df[df["topic_id"] == topic_id]
    if len(topic_data) == 0:
        print(f"Topic {topic_id} not found")
        return None
    
    row = topic_data.iloc[0]
    print("=" * 80)
    print(f"TOPIC {topic_id} DETAILS")
    print("=" * 80)
    print(f"\nKeywords: {row['keywords']}")
    print(f"Label: {row['label']}")
    print(f"\nTaxonomy:")
    print(f"  Main Category: {row['taxonomy_main_name']} (ID: {row['taxonomy_main_id']})")
    print(f"  Category Group: {row['taxonomy_main_group']}")
    print(f"  Secondary Category: {row['taxonomy_secondary_name']}")
    print(f"  Confidence: {row['taxonomy_confidence']}")
    print(f"  Is Noise: {row['taxonomy_is_noise']}")
    print(f"\nRadway:")
    print(f"  Main Function: {row['radway_main_name']} (ID: {row['radway_main_id']})")
    print(f"  Phase: {row['radway_phase_name']} (Phase: {row['radway_phase']})")
    print(f"  Secondary ID: {row['radway_secondary_id']}")
    print(f"  Confidence: {row['radway_confidence']}")
    print(f"  Is None: {row['radway_is_none']}")
    if pd.notna(row['radway_rationale']):
        print(f"  Rationale: {row['radway_rationale']}")
    
    return row

# Example: Explore a specific topic
example_topic_id = df["topic_id"].iloc[0]
explore_topic(df, example_topic_id)


TOPIC 0 DETAILS

Keywords: ita, ia, need, say, want, going, dona, means, think, idea
Label: Negotiating Deal

Taxonomy:
  Main Category: Hero's Elite Work & Business World (ID: 6.1)
  Category Group: Work, Wealth, Status & Institutions
  Secondary Category: Bonding, Everyday Intimacy & Growth
  Confidence: medium
  Is Noise: False

Radway:
  Main Function: None of the above (ID: none)
  Phase: Not a narrative function (Phase: NA)
  Secondary ID: None
  Confidence: high
  Is None: True
  Rationale: This topic is about the hero's work and business world, which is background context for the heroine–hero relationship. It does not primarily realise any of Radway's 13 functions.


topic_id                                                                    0
keywords                    ita, ia, need, say, want, going, dona, means, ...
num_keywords                                                               27
all_keywords                [ita, ia, need, say, want, going, dona, means,...
label                                                        Negotiating Deal
scene_summary               The couple discusses terms and expectations fo...
primary_categories                       relationship_conflict, domestic_life
secondary_categories                 setting:living_room, activity:discussion
label_is_noise                                                          False
label_rationale             The top keywords 'means', 'deal', 'today' and ...
taxonomy_main_id                                                          6.1
taxonomy_main_name                         Hero's Elite Work & Business World
taxonomy_main_group                       Work, Wealth, Status &

In [32]:
# Find topics with specific combinations
def find_topics_by_combination(df, taxonomy_category=None, radway_function=None, radway_phase=None):
    """Find topics matching specific taxonomy and Radway combinations."""
    mask = pd.Series([True] * len(df))
    if taxonomy_category:
        mask = mask & (df["taxonomy_main_name"] == taxonomy_category)
    if radway_function:
        mask = mask & (df["radway_main_name"] == radway_function)
    if radway_phase:
        mask = mask & (df["radway_phase_name"] == radway_phase)
    return df[mask]

# Example: Find topics with a specific combination
# Replace with actual values from your data
if len(df) > 0:
    example_tax = df["taxonomy_main_name"].value_counts().index[0]
    example_radway = df[df["radway_is_none"] == False]["radway_main_name"].value_counts().index[0] if len(df[df["radway_is_none"] == False]) > 0 else None
    
    if example_radway:
        combined_df = find_topics_by_combination(df, taxonomy_category=example_tax, radway_function=example_radway)
        print(f"Topics with taxonomy '{example_tax}' and Radway function '{example_radway}': {len(combined_df)}")
        if len(combined_df) > 0:
            combined_df[["topic_id", "keywords", "taxonomy_main_name", "radway_main_name", "radway_phase_name"]].head()


Topics with taxonomy 'Conflict, Distance & Breakup Threats' and Radway function 'Hero and heroine are physically or emotionally separated': 49


## 10. Advanced Visualizations

In [33]:
# Sankey diagram: Taxonomy Group -> Radway Phase
# This shows the flow from taxonomy groups to Radway phases
crosstab = pd.crosstab(df["taxonomy_main_group"], df["radway_phase_name"])
crosstab_normalized = crosstab.div(crosstab.sum(axis=1), axis=0)

# Create source, target, and value lists for Sankey
source = []
target = []
value = []
label = list(crosstab.index) + list(crosstab.columns)

for i, group in enumerate(crosstab.index):
    for j, phase in enumerate(crosstab.columns):
        if crosstab.loc[group, phase] > 0:
            source.append(i)
            target.append(len(crosstab.index) + j)
            value.append(crosstab.loc[group, phase])

fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=label,
    ),
    link=dict(
        source=source,
        target=target,
        value=value,
    )
)])

fig.update_layout(title_text="Taxonomy Group → Radway Phase Flow", font_size=10, height=600)
fig.show()

In [34]:
# Distribution of topics across taxonomy groups and Radway phases
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Taxonomy Groups", "Radway Phases"),
    specs=[[{"type": "pie"}, {"type": "pie"}]]
)

taxonomy_counts = df["taxonomy_main_group"].value_counts()
radway_counts = df["radway_phase_name"].value_counts()

fig.add_trace(
    go.Pie(labels=taxonomy_counts.index, values=taxonomy_counts.values, name="Taxonomy"),
    row=1, col=1
)

fig.add_trace(
    go.Pie(labels=radway_counts.index, values=radway_counts.values, name="Radway"),
    row=1, col=2
)

fig.update_traces(hole=0.4, hoverinfo="label+percent+name")
fig.update_layout(title_text="Topic Distribution: Taxonomy Groups vs Radway Phases")
fig.show()

In [35]:
# Scatter plot: Taxonomy confidence vs Radway confidence
# First, convert confidence to numeric for plotting
confidence_map = {"low": 1, "medium": 2, "high": 3}
df_plot = df.copy()
df_plot["taxonomy_conf_num"] = df_plot["taxonomy_confidence"].map(confidence_map)
df_plot["radway_conf_num"] = df_plot["radway_confidence"].map(confidence_map)

fig = px.scatter(
    df_plot,
    x="taxonomy_conf_num",
    y="radway_conf_num",
    color="taxonomy_main_group",
    size_max=10,
    hover_data=["topic_id", "keywords", "taxonomy_main_name", "radway_main_name"],
    title="Taxonomy Confidence vs Radway Confidence",
    labels={
        "taxonomy_conf_num": "Taxonomy Confidence (1=low, 2=medium, 3=high)",
        "radway_conf_num": "Radway Confidence (1=low, 2=medium, 3=high)"
    }
)
fig.update_traces(marker=dict(size=8))
fig.show()

## 11. Export and Save Results

In [36]:
# Export full DataFrame to CSV and Parquet
output_dir = project_root / "results" / "stage09_category_mapping" / "stage3_radway_functions" / "eda"
output_dir.mkdir(parents=True, exist_ok=True)

csv_path = output_dir / "full_model_data.csv"
parquet_path = output_dir / "full_model_data.parquet"

df.to_csv(csv_path, index=False)
df.to_parquet(parquet_path, index=False)

print(f"✓ Exported full data to:")
print(f"  - CSV: {csv_path}")
print(f"  - Parquet: {parquet_path}")

✓ Exported full data to:
  - CSV: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage09_category_mapping/stage3_radway_functions/eda/full_model_data.csv
  - Parquet: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage09_category_mapping/stage3_radway_functions/eda/full_model_data.parquet


In [37]:
# Save summary statistics
summary_serializable = {k: int(v) if isinstance(v, (np.integer, int)) else v for k, v in summary.items()}
with open(output_dir / "summary_statistics.json", "w") as f:
    json.dump(summary_serializable, f, indent=2)
print(f"✓ Saved summary statistics to {output_dir / 'summary_statistics.json'}")

✓ Saved summary statistics to /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage09_category_mapping/stage3_radway_functions/eda/summary_statistics.json


## 12. Custom Queries and Analysis

Use the cells below to create custom queries and analyses based on your research questions.

In [38]:
# Example: Find all topics in a specific Radway phase with high confidence
def analyze_phase_topics(df, phase_name, min_confidence="high"):
    """Analyze topics in a specific Radway phase."""
    phase_df = df[df["radway_phase_name"] == phase_name]
    if min_confidence:
        conf_order = {"low": 1, "medium": 2, "high": 3}
        min_conf_num = conf_order.get(min_confidence, 3)
        phase_df = phase_df[phase_df["radway_confidence"].map(conf_order) >= min_conf_num]
    
    print(f"Topics in phase '{phase_name}' with confidence >= {min_confidence}: {len(phase_df)}")
    print(f"\nTaxonomy distribution:")
    print(phase_df["taxonomy_main_group"].value_counts())
    print(f"\nTop taxonomy categories:")
    print(phase_df["taxonomy_main_name"].value_counts().head(10))
    return phase_df

# Example usage - replace with your phase of interest
if len(df) > 0:
    phases = df["radway_phase_name"].unique()
    if len(phases) > 0:
        example_phase = phases[0]
        phase_analysis = analyze_phase_topics(df, example_phase)
        phase_analysis[["topic_id", "keywords", "taxonomy_main_name", "radway_main_name", "radway_confidence"]].head()

Topics in phase 'Not a narrative function' with confidence >= high: 108

Taxonomy distribution:
taxonomy_main_group
Spaces, Time, Activities & Objects     34
Work, Wealth, Status & Institutions    24
Emotions, Cognition & Inner Life       19
Social World Outside Couple            17
Embodied & Sensory Experience           8
Conflict, Risk & Harm                   5
Special                                 1
Name: count, dtype: int64

Top taxonomy categories:
taxonomy_main_name
Domestic Spaces & Routines                      21
Hero's Elite Work & Business World              14
Public & Leisure Spaces                         11
Friends & Social Circles                         9
Negative Emotions & Distress                     8
Positive Emotions & Security                     7
Exercise & Physical Activity                     6
Family & Kinship                                 5
Shared Workplaces & Professional Interaction     4
Community, Norms & Social Events                 3
Name: cou

In [39]:
# Example: Compare topics across different taxonomy groups
def compare_taxonomy_groups(df, group_names=None):
    """Compare Radway distributions across taxonomy groups."""
    if group_names is None:
        group_names = df["taxonomy_main_group"].value_counts().head(5).index.tolist()
    
    comparison_data = []
    for group in group_names:
        group_df = df[df["taxonomy_main_group"] == group]
        radway_dist = group_df["radway_main_name"].value_counts()
        for radway_func, count in radway_dist.items():
            comparison_data.append({
                "taxonomy_group": group,
                "radway_function": radway_func,
                "count": count
            })
    
    comp_df = pd.DataFrame(comparison_data)
    
    # Create interactive visualization
    fig = px.bar(
        comp_df,
        x="taxonomy_group",
        y="count",
        color="radway_function",
        title="Radway Function Distribution by Taxonomy Group",
        labels={"count": "Number of Topics", "taxonomy_group": "Taxonomy Group"},
        barmode="group",
        height=500
    )
    fig.update_xaxes(tickangle=45)
    fig.show()
    
    return comp_df

# Example usage
if len(df) > 0:
    comparison = compare_taxonomy_groups(df)
    comparison.head(20)

In [40]:
# Example: Find topics with mismatched taxonomy and Radway (interesting edge cases)
def find_mismatched_topics(df):
    """Find topics where taxonomy group and Radway phase seem mismatched."""
    # This is a heuristic - you can customize the logic
    # Example: topics in "emotional" taxonomy but in "conflict" phase
    mismatched = []
    
    # You can define your own mismatch criteria here
    # For example, find topics with low confidence in both
    low_conf_both = df[
        (df["taxonomy_confidence"] == "low") & 
        (df["radway_confidence"] == "low")
    ]
    
    print(f"Topics with low confidence in both taxonomy and Radway: {len(low_conf_both)}")
    return low_conf_both

mismatched_topics = find_mismatched_topics(df)
if len(mismatched_topics) > 0:
    mismatched_topics[["topic_id", "keywords", "taxonomy_main_name", "radway_main_name", 
                      "taxonomy_confidence", "radway_confidence"]].head(10)

Topics with low confidence in both taxonomy and Radway: 0


## 13. Accessing Model Directly for Advanced Analysis

The BERTopic model object provides additional functionality for deeper analysis.

In [41]:
# Access topic representations directly from the model
def get_topic_keywords(model, topic_id, n_words=20):
    """Get top keywords for a topic from the model."""
    if topic_id in model.topic_representations_:
        keywords = model.topic_representations_[topic_id]
        return [kw[0] for kw in keywords[:n_words]]
    return []

# Example: Get detailed keywords for a topic
example_topic_id = df["topic_id"].iloc[0]
keywords = get_topic_keywords(model, example_topic_id, n_words=20)
print(f"Topic {example_topic_id} keywords:")
print(", ".join(keywords))


Topic 0 keywords:
ita, ia, need, say, want, going, dona, means, think, idea, thought, cana, says, promise, work, told, youa, help, better, really


In [42]:
# Check what other attributes the model has
print("Available model attributes:")
print("=" * 80)
for attr in dir(model):
    if not attr.startswith("_") and hasattr(model, attr):
        try:
            value = getattr(model, attr)
            if not callable(value):
                print(f"  {attr}: {type(value).__name__}")
        except:
            pass


Available model attributes:
  c_tf_idf_: csr_matrix
  calculate_probabilities: bool
  ctfidf_model: ClassTfidfTransformer
  custom_labels_: list
  embedding_model: SentenceTransformerBackend
  hdbscan_model: HDBSCAN
  language: NoneType
  low_memory: bool
  min_topic_size: int
  n_gram_range: tuple
  nr_topics: NoneType
  probabilities_: ndarray
  representation_model: dict
  representative_docs_: dict
  representative_images_: NoneType
  seed_topic_list: NoneType
  top_n_words: int
  topic_aspects_: dict
  topic_embeddings_: ndarray
  topic_labels_: dict
  topic_mapper_: TopicMapper
  topic_radway_: dict
  topic_representations_: dict
  topic_sizes_: Counter
  topic_taxonomy_: dict
  topics_: list
  umap_model: UMAP
  vectorizer_model: CountVectorizer
  verbose: bool
  zeroshot_min_similarity: float
  zeroshot_topic_list: NoneType


In [43]:
# Access taxonomy and Radway mappings directly
print("Taxonomy mappings available:", hasattr(model, "topic_taxonomy_"))
print("Radway mappings available:", hasattr(model, "topic_radway_"))

# Example: Get all topics with a specific taxonomy category
if hasattr(model, "topic_taxonomy_") and model.topic_taxonomy_:
    # Find topics with a specific taxonomy category
    target_category = df["taxonomy_main_name"].value_counts().index[0]
    topic_ids_with_category = [
        tid for tid, tax_data in model.topic_taxonomy_.items()
        if tax_data.get("main_category_name") == target_category
    ]
    print(f"\nTopic IDs with category '{target_category}': {len(topic_ids_with_category)}")
    print(f"First 10: {topic_ids_with_category[:10]}")


Taxonomy mappings available: True
Radway mappings available: True

Topic IDs with category 'Conflict, Distance & Breakup Threats': 58
First 10: [28, 32, 37, 52, 59, 65, 66, 87, 89, 98]


## 14. Export Filtered Views

Export specific filtered datasets for further analysis or reporting.


In [44]:
# Export topics by Radway phase
for phase in df["radway_phase_name"].unique():
    # Skip None/NaN values
    if pd.isna(phase) or phase is None:
        phase_clean = "unknown"
        phase_display = "Unknown/None"
    else:
        phase_clean = phase.replace(" ", "_").replace("/", "_").lower()
        phase_display = phase
    
    phase_df = df[df["radway_phase_name"] == phase]
    phase_path = output_dir / f"topics_phase_{phase_clean}.csv"
    phase_df.to_csv(phase_path, index=False)
    print(f"✓ Exported {len(phase_df)} topics for phase '{phase_display}' to {phase_path.name}")


✓ Exported 108 topics for phase 'Not a narrative function' to topics_phase_not_a_narrative_function.csv
✓ Exported 20 topics for phase 'Commitment & Restoration' to topics_phase_commitment_&_restoration.csv
✓ Exported 170 topics for phase 'Initial Conflict & Isolation' to topics_phase_initial_conflict_&_isolation.csv
✓ Exported 63 topics for phase 'Turning Point & Recognition' to topics_phase_turning_point_&_recognition.csv
✓ Exported 0 topics for phase 'Unknown/None' to topics_phase_unknown.csv


In [45]:
# Export topics by taxonomy group
for group in df["taxonomy_main_group"].unique():
    if pd.notna(group):
        group_df = df[df["taxonomy_main_group"] == group]
        group_clean = str(group).replace(" ", "_").replace("/", "_").lower()
        group_path = output_dir / f"topics_taxonomy_group_{group_clean}.csv"
        group_df.to_csv(group_path, index=False)
        print(f"✓ Exported {len(group_df)} topics for taxonomy group '{group}' to {group_path.name}")

✓ Exported 27 topics for taxonomy group 'Work, Wealth, Status & Institutions' to topics_taxonomy_group_work,_wealth,_status_&_institutions.csv
✓ Exported 57 topics for taxonomy group 'Sexuality, Attraction & Intimacy' to topics_taxonomy_group_sexuality,_attraction_&_intimacy.csv
✓ Exported 18 topics for taxonomy group 'Social World Outside Couple' to topics_taxonomy_group_social_world_outside_couple.csv
✓ Exported 126 topics for taxonomy group 'Relationship Trajectory (Main Couple)' to topics_taxonomy_group_relationship_trajectory_(main_couple).csv
✓ Exported 63 topics for taxonomy group 'Emotions, Cognition & Inner Life' to topics_taxonomy_group_emotions,_cognition_&_inner_life.csv
✓ Exported 34 topics for taxonomy group 'Spaces, Time, Activities & Objects' to topics_taxonomy_group_spaces,_time,_activities_&_objects.csv
✓ Exported 9 topics for taxonomy group 'Embodied & Sensory Experience' to topics_taxonomy_group_embodied_&_sensory_experience.csv
✓ Exported 26 topics for taxonomy gro

In [46]:
# Export high-confidence topics only
high_conf_df = df[
    (df["taxonomy_confidence"] == "high") & 
    (df["radway_confidence"] == "high")
]
high_conf_path = output_dir / "topics_high_confidence.csv"
high_conf_df.to_csv(high_conf_path, index=False)
print(f"✓ Exported {len(high_conf_df)} high-confidence topics to {high_conf_path.name}")

✓ Exported 154 high-confidence topics to topics_high_confidence.csv


## 15. Summary and Next Steps

This notebook provides comprehensive interactive exploration of the BERTopic model with Radway mappings. Key capabilities include:

1. **Data Overview**: Summary statistics and data structure exploration
2. **Taxonomy Analysis**: Distribution of taxonomy categories and groups
3. **Radway Analysis**: Distribution of narrative functions and phases
4. **Cross-Analysis**: Relationships between taxonomy and Radway mappings
5. **Interactive Filtering**: Filter by category, function, phase, confidence, keywords
6. **Topic Deep Dives**: Detailed examination of individual topics
7. **Advanced Visualizations**: Sankey diagrams, scatter plots, heatmaps
8. **Custom Queries**: Build custom analyses for specific research questions
9. **Data Export**: Export full dataset or filtered views

### Tips for Further Exploration

- Modify the filtering functions to create custom queries
- Use the interactive Plotly visualizations to explore relationships
- Export filtered datasets for external analysis
- Access the model object directly for advanced BERTopic functionality
- Combine multiple filters to find specific topic patterns

### Notes

- All data is exported to: `results/stage09_category_mapping/stage3_radway_functions/eda/`
- The model can be reloaded using the `load_model_with_radway()` function
- Customize the notebook cells to answer your specific research questions

In [47]:
# Final summary
print("=" * 80)
print("NOTEBOOK SUMMARY")
print("=" * 80)
print(f"\nTotal topics analyzed: {len(df)}")
print(f"Topics with taxonomy mappings: {df['taxonomy_main_id'].notna().sum()}")
print(f"Topics with Radway mappings: {df['radway_main_id'].notna().sum()}")
print(f"Topics with Radway functions: {(df['radway_is_none'] == False).sum()}")
print(f"\nUnique taxonomy categories: {df['taxonomy_main_name'].nunique()}")
print(f"Unique Radway functions: {df[df['radway_is_none'] == False]['radway_main_name'].nunique()}")
print(f"Unique Radway phases: {df['radway_phase_name'].nunique()}")
print(f"\nAll results saved to: {output_dir}")
print("\n✓ Interactive EDA complete!")

NOTEBOOK SUMMARY

Total topics analyzed: 368
Topics with taxonomy mappings: 361
Topics with Radway mappings: 361
Topics with Radway functions: 260

Unique taxonomy categories: 30
Unique Radway functions: 13
Unique Radway phases: 4

All results saved to: /home/polina/Documents/goodreads_romance_research_cursor/billionaire_novels_rating_predictor/results/stage09_category_mapping/stage3_radway_functions/eda

✓ Interactive EDA complete!
